In [8]:
# Contrôle final de la correspondance entre les demandes ctb / digifor
import re
import pandas as pd
from unidecode import unidecode
from datetime import datetime
from collections import Counter

date = datetime.now()
date = date.strftime("%d-%m-%Y")
digifor_path = r"C:\Users\L14\Desktop\Publicite\MAJ GESTIONNAIRE.xlsx"
ctb_path = r"C:\Users\L14\Downloads\tonkpi_plle_info_12-02-2026.xlsx"

def normalize(name):
    return str(name).lower().strip()

def format_name(name):
    def normalize(name):
        return '_'.join(str(name).lower().replace(';'," ").split()).strip()
    
    values = name.split(',',2)
    if len(values) > 1 :
        if normalize(values[0]) == normalize(values[1]):
            return values[0]
        else :
            return " ".join(values[:2])
    else :
        return name.replace(','," ")

def compare_name(row):
    if Counter(row['fullname_control'].split()) == Counter(row['obs_control'].split()):
        return True
    else :
        return False

# Lecture des fichiers excels pour les enquetes de DIGIFOR et les parcelles deCTB
digifor_demandes = pd.read_excel(digifor_path, engine='openpyxl')
ctb_parcelles = pd.read_excel(ctb_path, engine='openpyxl')

# Formatage du numéro de parcelle pour corriger les erreurs de saisie
digifor_demandes['numParcelleOF_c'] = (digifor_demandes['code_parcelle'].astype(str)
                                    .str.strip()
                                    .str.upper()
                                    .str.replace(' ','')
                                    .str.replace(':','')
                                    .str.replace('O','0'))

digifor_demandes['numParcelleOF_c'] = (digifor_demandes['numParcelleOF_c'].astype(str)
                                    .str.lstrip('P')
                                    .radd('P')
                                    .str.replace(r'^(P\d+)([A-Za-z]?)?.*$', lambda m: m.group(1) + (m.group(2) if len(m.group(2) or '') == 1 else ''), regex=True))

digifor_demandes['fullname'] = (digifor_demandes['fullname'].astype(str)
                                       .str.replace(r"[/:\-_.()°%]","",regex=True))
# Formatage du nom des demandeurs au niveau des parcelles
ctb_parcelles['obs'] = (ctb_parcelles['obs'].astype(str)
                                    .str.lower()
                                    .str.replace(r"[/:\-_.()°⁰%*]","",regex=True)
                                    .str.replace(" du ","")
                                    .str.replace(r"[A-Za-z]+[0-9]","",regex=True)
                                    .str.replace(r"[0-9][A-Za-z]+","",regex=True)
                                    .str.replace(r"(\D)(ci\d+)",lambda m: m.group(1),regex=True)
                                    .str.replace(r"(\D)(c\d+)",lambda m: m.group(1),regex=True)
                                    .str.replace(r"(\D)(n\d+)",lambda m: m.group(1),regex=True)
                                    .str.replace(r"(\D)(bf\d+)",lambda m: m.group(1),regex=True)
                                    .str.replace(r"(\D)(\d+)",lambda m: m.group(1),regex=True)
                                    .str.replace(r"(\D)(;\d+)",lambda m: m.group(1),regex=True)
                                    .str.strip())

ctb_parcelles["obs"] = ctb_parcelles["obs"].apply(format_name)
# Creation du champ pour le contrôle des noms
ctb_parcelles['obs_control'] = ctb_parcelles['obs'].astype(str).apply(lambda x : unidecode(normalize(x)))

#print(digifor_demandes.shape)
mask_annule = (digifor_demandes['fullname'].str.contains('annule', case=False, na=False) | digifor_demandes['code_parcelle'].str.contains('annule',case=False, na=False))
mask_erreur = (digifor_demandes['fullname'].str.contains('erreur', case=False, na=False) | digifor_demandes['code_parcelle'].str.contains('erreur', case=False, na=False))

digifor_demandes_annulees = digifor_demandes[(mask_annule | mask_erreur)]
digifor_demandes = digifor_demandes[~(mask_annule | mask_erreur)]

digifor_demandes['fullname_control'] = digifor_demandes['fullname'].astype(str).apply(lambda x : unidecode(normalize(x)))

digifor_demandes['code_par_digifor'] = digifor_demandes['code_vil'] + '-' + digifor_demandes['numParcelleOF_c']
ctb_parcelles['code_par_ctb'] = ctb_parcelles['cd_vil'] + '-' + ctb_parcelles['code_parcelle']

digifor_demandes_duplicated = digifor_demandes[digifor_demandes.duplicated(subset='code_par_digifor', keep=False)]
ctb_parcelles_duplicated = ctb_parcelles[ctb_parcelles.duplicated(subset='code_par_ctb', keep=False)]

digifor_demandes = digifor_demandes.drop_duplicates(subset='code_par_digifor')
ctb_parcelles = ctb_parcelles.drop_duplicates(subset='code_par_ctb')


ctb_parcelles_digifor_demandes = pd.merge(digifor_demandes,ctb_parcelles,left_on='code_par_digifor',right_on='code_par_ctb',how='outer', indicator=True)
ctb_parcelles_digifor_demandes_inner = ctb_parcelles_digifor_demandes[ctb_parcelles_digifor_demandes['_merge'] == 'both']
ctb_parcelles_digifor_demandes_left = ctb_parcelles_digifor_demandes[ctb_parcelles_digifor_demandes['_merge'] == 'left_only']
ctb_parcelles_digifor_demandes_right = ctb_parcelles_digifor_demandes[ctb_parcelles_digifor_demandes['_merge'] == 'right_only']

ctb_parcelles_digifor_demandes_inner['same_name'] = ctb_parcelles_digifor_demandes_inner.apply(compare_name, axis=1)
ctb_parcelles_digifor_demandes_inner_same_name = ctb_parcelles_digifor_demandes_inner[ctb_parcelles_digifor_demandes_inner['same_name']]
ctb_parcelles_digifor_demandes_inner_diff_name = ctb_parcelles_digifor_demandes_inner[~ctb_parcelles_digifor_demandes_inner['same_name']]

#print(digifor_demandes_annulees.shape)
print(digifor_demandes.shape)
print(ctb_parcelles.shape)
print(ctb_parcelles_digifor_demandes.shape)

ctb_parcelles_duplicated.to_excel(r"C:\Users\L14\Desktop\Publicite\doublons_ctb_parcelles.xlsx")
digifor_demandes_duplicated.to_excel(r"C:\Users\L14\Desktop\Publicite\doublons_digifor_parcelles.xlsx")
ctb_parcelles_digifor_demandes_inner_same_name.to_excel(r"C:\Users\L14\Desktop\Publicite\ctb_parcelles_digifor_demandes_inner_same_name.xlsx")
ctb_parcelles_digifor_demandes_inner_diff_name.to_excel(r"C:\Users\L14\Desktop\Publicite\ctb_parcelles_digifor_demandes_inner_diff_name.xlsx")


C:\Users\L14\AppData\Local\Temp\ipykernel_30040\1898377186.py:97: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ctb_parcelles_digifor_demandes_inner['same_name'] = ctb_parcelles_digifor_demandes_inner.apply(compare_name, axis=1)


(4780, 23)
(6257, 11)
(6263, 35)
